# Bedrock Client 2: Video & Image Generation

This notebook demonstrates:
- 🎨 Image generation with Amazon Nova Canvas (InvokeModel API)
- 💬 Text generation with Amazon Nova (Converse API)
- 🛡️ Error handling scenarios

# Initialization and Configuration


In [ ]:
import json
import base64
import io
import time

import boto3
import requests
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError, BotoCoreError
from PIL import Image
import matplotlib.pyplot as plt



import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment-specific .env file (e.g., .env.dev, .env.test)
ENVIRONMENT = os.getenv("ENVIRONMENT", "dev")
env_file = Path(__file__).parent / f".env.{ENVIRONMENT}" if "__file__" in dir() else Path(f".env.{ENVIRONMENT}")
load_dotenv(env_file)

# Configuration
MODELS = {
    "image_gen": "amazon.nova-canvas-v1:0",
    "video_gen": "amazon.nova-reel-v1:0",
    "text_gen": "amazon.nova-lite-v1:0"
}
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_URL = os.getenv("GATEWAY_API_URL", "https://your-gateway-url.com")
SECRET_ID = os.getenv("GATEWAY_SECRET_ID", "bedrock-gateway-dev-oauth-credentials")

# Extract the secret value from Secret Manager

session = boto3.Session(profile_name=os.getenv("AWS_PROFILE"), region_name=AWS_REGION)
JSON_SECRET = json.loads(session.client('secretsmanager').get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

## Token Generation

In [ ]:
def generate_token():
    """Generate access token for API authentication"""
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
        "audience": "bedrockproxygateway",
        "scope": "bedrockproxygateway:invoke"
    }

    response = requests.post(TOKEN_URL, data=payload)
    response.raise_for_status()
    token = response.json()['access_token']
    return f"Bearer {token}"

def create_bedrock_client(token):
    """Create and configure Bedrock runtime client"""
    bedrock = boto3.client(
        'bedrock-runtime',
        region_name=AWS_REGION,
        endpoint_url=API_URL,
        config=Config(signature_version=UNSIGNED, retries={'max_attempts': 0}),
        verify=False
    )

    def add_api_token(request, **kwargs):
        request.headers.add_header('Authorization', token)
        return request
    
    bedrock.meta.events.register('before-sign.bedrock-runtime.*', add_api_token)
    return bedrock

## Bedrock Runtime Client Setup

In [ ]:
# Generate token
TOKEN = generate_token()
print(f"Token generated: {TOKEN[:20]}...")

# Create Bedrock client
bedrock = create_bedrock_client(TOKEN)
print("Bedrock client configured successfully")

## Helper Functions


In [ ]:
def display_base64_image(base64_string: str, title: str = "Generated Image"):
    """Display base64 encoded image in notebook"""
    try:
        image_data = base64.b64decode(base64_string)
        image = Image.open(io.BytesIO(image_data))
        plt.figure(figsize=(10, 8))
        plt.imshow(image)
        plt.title(title, fontsize=14)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"❌ Failed to display image: {e}")

def measure_performance(func):
    """Decorator to measure function execution time"""
    def wrapper(*args, **kwargs):
        start_time = time.time()
        try:
            result = func(*args, **kwargs)
            end_time = time.time()
            execution_time = end_time - start_time
            return result, execution_time
        except Exception as e:
            end_time = time.time()
            execution_time = end_time - start_time
            raise
    return wrapper

# Image Generation with Amazon Nova Canvas (InvokeModel API)

Testing Amazon Nova Canvas model for image generation using the InvokeModel API with performance monitoring.


In [ ]:
@measure_performance
def generate_image_with_nova_invoke(prompt: str, **config) -> str:
    """Generate image using Nova Canvas with InvokeModel API"""
    payload = {
        "taskType": "TEXT_IMAGE",
        "textToImageParams": {
            "text": prompt,
            "negativeText": config.get("negative_text", "blurry, low quality, distorted")
        },
        "imageGenerationConfig": {
            "numberOfImages": config.get("num_images", 1),
            "height": config.get("height", 512),
            "width": config.get("width", 512),
            "cfgScale": config.get("cfg_scale", 7.0),
            "seed": config.get("seed", 42),
            "quality": config.get("quality", "standard")
        }
    }
    try:
        response = bedrock.invoke_model(
            modelId=MODELS["image_gen"],
            body=json.dumps(payload)
        )
        result = json.loads(response['body'].read())
        if 'images' in result and len(result['images']) > 0:
            return result['images'][0]
        else:
            raise Exception("No images returned in response")
    except (ClientError, BotoCoreError) as e:
        error_code = getattr(e, 'response', {}).get('Error', {}).get('Code', 'Unknown')
        raise Exception(f"AWS Error ({error_code}): {str(e)}")
    except Exception as e:
        raise Exception(f"Unexpected error: {str(e)}")

# Test image generation with performance monitoring
print("🎨 Testing Image Generation with InvokeModel API")

image_configs = [
    {
        "prompt": "A futuristic electric car in dallas city at night with logo AWS, neon lights reflecting on wet streets",
        "cfg_scale": 7.0,
        "seed": 42
    },
    {
        "prompt": "A minimalist mountain landscape, clean lines, pastel colors",
        "cfg_scale": 5.0,
        "seed": 123
    }
]

for i, config in enumerate(image_configs, 1):
    try:
        print(f"\n🎯 Generating image {i}: {config['prompt'][:50]}...")
        image_b64, exec_time = generate_image_with_nova_invoke(
            config["prompt"],
            cfg_scale=config["cfg_scale"],
            seed=config["seed"]
        )
        print(f"✅ Image {i} generated in {exec_time:.2f}s")
        display_base64_image(image_b64, f"Nova Canvas - Image {i}")
    except Exception as e:
        print(f"❌ Image {i} generation failed: {e}")
    except Exception as e:
        print(f"❌ Unexpected error for image {i}: {e}")

# Text Generation with Amazon Nova (Converse API)

Testing Amazon Nova lite text generation using the Converse API.

In [ ]:
@measure_performance
def generate_text_with_nova_converse(prompt: str, **config) -> str:
    """Generate text using Nova with Converse API"""
    try:
        response = bedrock.converse(
            modelId=MODELS["text_gen"],
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            inferenceConfig={
                "maxTokens": config.get("max_tokens", 500),
                "temperature": config.get("temperature", 0.7),
                "topP": config.get("top_p", 0.9)
            }
        )
        if 'output' in response and 'message' in response['output']:
            content = response['output']['message']['content']
            if content and len(content) > 0 and 'text' in content[0]:
                return content[0]['text']
        raise Exception("No text returned in response")
    except (ClientError, BotoCoreError) as e:
        error_code = getattr(e, 'response', {}).get('Error', {}).get('Code', 'Unknown')
        raise Exception(f"AWS Error ({error_code}): {str(e)}")
    except Exception as e:
        raise Exception(f"Unexpected error: {str(e)}")

# Test text generation with Converse API
print("💬 Testing Text Generation with Converse API")

text_prompts = [
    {
        "prompt": "Explain the benefits of electric vehicles in 3 bullet points",
        "temperature": 0.3
    },
    {
        "prompt": "Write a creative short story about a robot learning to paint",
        "temperature": 0.8,
        "max_tokens": 300
    }
]

for i, config in enumerate(text_prompts, 1):
    try:
        print(f"\n🎯 Generating text {i}: {config['prompt'][:50]}...")
        text_response, exec_time = generate_text_with_nova_converse(
            config["prompt"],
            temperature=config.get("temperature", 0.7),
            max_tokens=config.get("max_tokens", 500)
        )
        print(f"✅ Text {i} generated in {exec_time:.2f}s")
        print(f"📝 Response:\n{text_response}\n")
    except Exception as e:
        print(f"❌ Text {i} generation failed: {e}")
    except Exception as e:
        print(f"❌ Unexpected error for text {i}: {e}")

# Performance Tuning and Optimization

Testing different configurations for optimal performance.


In [ ]:
# Performance comparison tests
print("⚡ Performance Tuning Tests")

# Test different image sizes and their impact on generation time
size_tests = [
    {"width": 320, "height": 320, "name": "Small"},
    {"width": 512, "height": 512, "name": "Medium"},
    {"width": 768, "height": 768, "name": "Large"}
]

performance_results = []
test_prompt = "A simple geometric shape, minimalist design"

for size_config in size_tests:
    try:
        print(f"\n📏 Testing {size_config['name']} size ({size_config['width']}x{size_config['height']})")
        image_b64, exec_time = generate_image_with_nova_invoke(
            test_prompt,
            width=size_config["width"],
            height=size_config["height"],
            cfg_scale=5.0,
            seed=42
        )
        image_size_kb = len(image_b64) * 3 / 4 / 1024  # Approximate KB from base64
        result = {
            "size_name": size_config["name"],
            "dimensions": f"{size_config['width']}x{size_config['height']}",
            "generation_time": exec_time,
            "image_size_kb": image_size_kb
        }
        performance_results.append(result)
        print(f"✅ Generated in {exec_time:.2f}s, Size: {image_size_kb:.1f}KB")
    except Exception as e:
        print(f"❌ Failed {size_config['name']} test: {e}")

# Display performance summary
print("\n📊 Performance Summary:")
print("Size\t\tDimensions\tTime (s)\tSize (KB)")
print("-" * 50)
for result in performance_results:
    print(f"{result['size_name']}\t\t{result['dimensions']}\t\t{result['generation_time']:.2f}\t\t{result['image_size_kb']:.1f}")

# Performance recommendations
if performance_results:
    fastest = min(performance_results, key=lambda x: x['generation_time'])
    print(f"\n🏆 Fastest configuration: {fastest['size_name']} ({fastest['generation_time']:.2f}s)")
    print("💡 Recommendations:")
    print("   - Use smaller dimensions for faster generation")
    print("   - Lower CFG scale (5.0-7.0) for better speed/quality balance")